# SAP - Sdružení automobilového průmyslu

This notebook ingests vehicle production and sales data from **SAP** (Sdružení automobilového průmyslu / Czech Automotive Industry Association).

## Source
- **Location:** `/Volumes/agentbricks/sector_data_raw/sap_data`
- **Format:** xlsx files (monthly production reports, electric vehicle reports, yearly time series)

## Tables Created

| Table | Description |
| --- | --- |
| `agentbricks.sector_data_bronze.sap_yearly_production_sales` | Annual production, domestic sales, and export by vehicle category (1996–2025) |
| `agentbricks.sector_data_bronze.sap_monthly_production` | Monthly production, domestic sales, and export of passenger cars **per brand** (ŠKODA AUTO, Hyundai, Toyota). No aggregated totals. |
| `agentbricks.sector_data_bronze.sap_ev_production_monthly` | Monthly electric vehicle production **per brand** (ŠKODA AUTO, Hyundai) by powertrain (BEV, PHEV). No aggregated totals. |

## Key Design Decisions
- **No "Celkem" (totals)** — only per-brand data; users can aggregate themselves
- **Monthly values only** — cumulative YTD figures are excluded; only single-month production counts
- Data extracted from the "single month" rows in each xlsx file (not the "leden-březen" cumulative rows)

In [0]:
%pip install openpyxl --quiet

In [0]:
import os
import re
import openpyxl
import time
from datetime import date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql import functions as F

SOURCE_PATH = "/Volumes/agentbricks/sector_data_raw/sap_data"
TARGET_SCHEMA = "agentbricks.sector_data_bronze"

print(f"Source: {SOURCE_PATH}")
print(f"Target schema: {TARGET_SCHEMA}")

In [0]:
# =============================================================
# TABLE 1: Yearly production, domestic sales, export (1996-2025)
# Source: rocni-casove-rady-vyroby-a-odbytu-vozidel-2025.xlsx
# =============================================================

TARGET_TABLE_YEARLY = f"{TARGET_SCHEMA}.sap_yearly_production_sales"

yearly_file = os.path.join(SOURCE_PATH, "rocni-casove-rady-vyroby-a-odbytu-vozidel-2025.xlsx")
wb = openpyxl.load_workbook(yearly_file, read_only=True, data_only=True)
ws = wb[wb.sheetnames[0]]
rows = list(ws.iter_rows(values_only=True))
wb.close()

# Row 4 has years in columns 2-30 (1996-2025)
year_row = rows[4]
years = [int(y) for y in year_row[2:32] if y is not None and str(y).isdigit()]

# Vehicle categories and their row positions
# Format: (category_name, start_row) - each category has 3 rows: Výroba, Tuzemský prodej, Export
categories = [
    ("Osobní automobily a LUV (M1, N1)", 6),
    ("Autobusy (M2, M3)", 10),
    ("Motocykly (L)", 14),
    ("Nákladní vozidla (N2, N3)", 18),
    ("Přípojná vozidla (O1, O2)", 22),
    ("Přípojná vozidla (O3, O4)", 26),
]

metrics = ["Výroba", "Tuzemský prodej", "Export"]

yearly_records = []
for category, start_row in categories:
    for metric_idx, metric in enumerate(metrics):
        row = rows[start_row + metric_idx]
        for year_idx, year in enumerate(years):
            col_idx = year_idx + 2  # data starts at column 2
            val = row[col_idx]
            if val is not None and val != 'n/a':
                try:
                    yearly_records.append((year, category, metric, int(val)))
                except (ValueError, TypeError):
                    pass

print(f"Yearly records parsed: {len(yearly_records):,}")
print(f"Years: {min(r[0] for r in yearly_records)} - {max(r[0] for r in yearly_records)}")
print(f"Categories: {len(categories)}")

In [0]:
# Write yearly table
yearly_schema = StructType([
    StructField("year", IntegerType(), False),
    StructField("vehicle_category", StringType(), False),
    StructField("metric", StringType(), False),
    StructField("value", IntegerType(), False),
])

df_yearly = spark.createDataFrame(yearly_records, schema=yearly_schema)

df_yearly.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE_YEARLY)

spark.sql(f"""COMMENT ON TABLE {TARGET_TABLE_YEARLY} IS
    'Annual vehicle production, domestic sales, and export in Czech Republic by vehicle category. Source: SAP (Sdru\u017een\u00ed automobilov\u00e9ho pr\u016fmyslu). Data from 1996 onwards.'""")
spark.sql(f"ALTER TABLE {TARGET_TABLE_YEARLY} ALTER COLUMN year COMMENT 'Reporting year'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_YEARLY} ALTER COLUMN vehicle_category COMMENT 'Vehicle category (Osobn\u00ed automobily, Autobusy, Motocykly, N\u00e1kladn\u00ed vozidla, P\u0159\u00edpojn\u00e1 vozidla)'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_YEARLY} ALTER COLUMN metric COMMENT 'Metric type: V\u00fdroba (production), Tuzemsk\u00fd prodej (domestic sales), Export'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_YEARLY} ALTER COLUMN value COMMENT 'Number of vehicles'")

print(f"\u2705 {TARGET_TABLE_YEARLY}: {spark.table(TARGET_TABLE_YEARLY).count():,} rows")
display(spark.table(TARGET_TABLE_YEARLY).filter("vehicle_category LIKE 'Osobn\u00ed%'").orderBy(F.desc("year")).limit(10))

In [0]:
# =============================================================
# TABLE 2: Monthly production/sales/export PER BRAND
# Source: ALL vyroba-a-odbyt-vozidel-M-YYYY.xlsx files
# Extracts only per-brand data (no "Celkem" aggregates)
# =============================================================

TARGET_TABLE_MONTHLY = f"{TARGET_SCHEMA}.sap_monthly_production"

def parse_sap_filename(filename: str) -> tuple:
    """Extract month and year from filename like 'vyroba-a-odbyt-vozidel-6-2025.xlsx'"""
    match = re.search(r'-(\d{1,2})-(\d{4})', filename)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

# ALL monthly/quarterly production files
all_prod_files = sorted([
    f for f in os.listdir(SOURCE_PATH)
    if f.startswith('vyroba-a-odbyt-vozidel-') and f.endswith('.xlsx')
])
print(f"Production files to process: {len(all_prod_files)}")

# Brand column positions: (brand_name, period_col, year_col, value_col)
# Each brand occupies 6 columns: period, year, value, diff, %, gap
BRAND_POSITIONS = [
    ("ŠKODA AUTO", 1, 2, 3),
    ("Hyundai", 7, 8, 9),
    ("Toyota", 13, 14, 15),
]

czech_months_list = ['leden', 'únor', 'březen', 'duben', 'květen', 'červen',
                    'červenec', 'srpen', 'září', 'říjen', 'listopad', 'prosinec']

monthly_records = []

for filename in all_prod_files:
    report_month, report_year = parse_sap_filename(filename)
    if report_month is None:
        continue
    
    filepath = os.path.join(SOURCE_PATH, filename)
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb[wb.sheetnames[0]]
    rows = list(ws.iter_rows(values_only=True))
    wb.close()
    
    report_date = date(report_year, report_month, 1)
    
    # Find brand header row (contains 'ŠKODA AUTO' at col 1)
    brand_header_idx = None
    for i, row in enumerate(rows):
        if row[1] is not None and 'ŠKODA' in str(row[1]).upper():
            brand_header_idx = i
            break
    
    if brand_header_idx is None:
        print(f"  {filename}: No brand section found, skipping")
        continue
    
    # Find the end of the passenger car brand section
    brand_section_end = len(rows)
    for i in range(brand_header_idx + 1, len(rows)):
        if rows[i][1] is not None and str(rows[i][1]).strip().upper() in ['AUTOBUSY', 'MOTOCYKLY', 'MOTOCYKLY ']:
            brand_section_end = i
            break
    
    # Scan for metric sections and monthly data within brand section
    # Structure: METRIC_HEADER -> Období header -> YTD row -> (empty) -> MONTH row
    # The month row has: period_name, year, value ON THE SAME ROW
    current_metric = None
    
    for i in range(brand_header_idx + 1, brand_section_end):
        row = rows[i]
        if row[1] is None:
            continue
        
        cell = str(row[1]).strip()
        cell_upper = cell.upper()
        
        # Detect metric header
        if 'VÝROBA' in cell_upper and 'OBDOB' not in cell_upper:
            current_metric = 'Výroba'
            continue
        elif 'TUZEMSK' in cell_upper:
            current_metric = 'Tuzemský prodej'
            continue
        elif cell_upper == 'EXPORT' or cell_upper == 'EXPORT ':
            current_metric = 'Export'
            continue
        
        # Check if this row is a single-month data row
        if current_metric and cell.lower() in czech_months_list:
            # This IS the data row - period, year, value are on THIS row
            for brand_name, period_col, year_col, value_col in BRAND_POSITIONS:
                try:
                    year_val = row[year_col]
                    count_val = row[value_col]
                    if year_val == report_year and count_val is not None:
                        val = int(count_val)
                        if val > 0:
                            monthly_records.append((
                                report_date, "Osobní automobily", brand_name,
                                current_metric, val
                            ))
                except (ValueError, TypeError, IndexError):
                    pass
            
            # Reset metric after extracting single-month row
            current_metric = None
    
    print(f"  {filename}: {len(monthly_records):,} total records")

print(f"\nTotal monthly records: {len(monthly_records):,}")

In [0]:
# Write monthly production table (per brand only, no Celkem)
monthly_schema = StructType([
    StructField("report_date", DateType(), False),
    StructField("vehicle_category", StringType(), False),
    StructField("brand", StringType(), False),
    StructField("metric", StringType(), False),
    StructField("value", IntegerType(), False),
])

df_monthly = spark.createDataFrame(monthly_records, schema=monthly_schema)
df_monthly = df_monthly.dropDuplicates()

df_monthly.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE_MONTHLY)

spark.sql(f"""COMMENT ON TABLE {TARGET_TABLE_MONTHLY} IS
    'Monthly passenger car production, domestic sales, and export in Czech Republic PER BRAND. No aggregated totals - only individual manufacturer data (ŠKODA AUTO, Hyundai, Toyota). Source: SAP (Sdružení automobilového průmyslu).'""")
spark.sql(f"ALTER TABLE {TARGET_TABLE_MONTHLY} ALTER COLUMN report_date COMMENT 'First day of the reporting month'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_MONTHLY} ALTER COLUMN vehicle_category COMMENT 'Vehicle category'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_MONTHLY} ALTER COLUMN brand COMMENT 'Manufacturer brand (ŠKODA AUTO, Hyundai, Toyota)'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_MONTHLY} ALTER COLUMN metric COMMENT 'Výroba (production), Tuzemský prodej (domestic sales), Export'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_MONTHLY} ALTER COLUMN value COMMENT 'Number of vehicles'")

print(f"\u2705 {TARGET_TABLE_MONTHLY}: {spark.table(TARGET_TABLE_MONTHLY).count():,} rows")
print(f"\nBrand breakdown:")
display(spark.table(TARGET_TABLE_MONTHLY).groupBy("brand", "metric").count().orderBy("brand", "metric"))
print(f"\nSample data:")
display(spark.table(TARGET_TABLE_MONTHLY).orderBy(F.desc("report_date"), "brand", "metric").limit(15))

In [0]:
# =============================================================
# TABLE 3: Electric vehicle production (BEV/PHEV) PER BRAND
# Source: vyroba-elektrickych-vozidel-M-YYYY.xlsx
# No "Celkem" - only per-brand data (ŠKODA AUTO, Hyundai)
# =============================================================

TARGET_TABLE_EV = f"{TARGET_SCHEMA}.sap_ev_production_monthly"

ev_files = sorted([
    f for f in os.listdir(SOURCE_PATH)
    if f.startswith('vyroba-elektrickych-vozidel-') and f.endswith('.xlsx')
])
print(f"EV files to process: {len(ev_files)}")

czech_months = ['leden', 'únor', 'březen', 'duben', 'květen', 'červen',
               'červenec', 'srpen', 'září', 'říjen', 'listopad', 'prosinec']

ev_records = []

for filename in ev_files:
    report_month, report_year = parse_sap_filename(filename)
    if report_month is None:
        continue
    
    filepath = os.path.join(SOURCE_PATH, filename)
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb[wb.sheetnames[0]]
    rows_data = list(ws.iter_rows(values_only=True))
    wb.close()
    
    report_date = date(report_year, report_month, 1)
    
    # Find brand section start (ŠKODA AUTO header after row 25)
    brand_section_start = None
    for i in range(25, len(rows_data)):
        if rows_data[i][1] is not None and 'ŠKODA' in str(rows_data[i][1]).upper():
            brand_section_start = i
            break
    
    if brand_section_start is None:
        continue
    
    # Find where AUTOBUSY section starts (end of passenger car brand section)
    autobusy_start = len(rows_data)
    for i in range(brand_section_start + 1, len(rows_data)):
        if rows_data[i][1] is not None and 'AUTOBUSY' in str(rows_data[i][1]).upper():
            autobusy_start = i
            break
    
    # Parse brands: ŠKODA AUTO at col 1, HYUNDAI at col 7
    brands_config = [
        ('ŠKODA AUTO', 1, 2),   # (name, year_col, value_col)
        ('Hyundai', 7, 8),
    ]
    
    for brand_name, year_col, val_col in brands_config:
        # Find the monthly sub-section for this brand (single month name, no hyphen)
        for i in range(brand_section_start, autobusy_start):
            row = rows_data[i]
            if row[year_col] is not None:
                cell = str(row[year_col]).strip().lower()
                # Single month name = monthly section (not YTD)
                if cell in czech_months and '-' not in cell:
                    # Capture BEV, PHEV from the next rows
                    metrics_found = 0
                    for j in range(i + 1, min(i + 12, autobusy_start)):
                        if rows_data[j][year_col] is not None and str(rows_data[j][year_col]).strip() in ['BEV', 'PHEV']:
                            pt = str(rows_data[j][year_col]).strip()
                            if j + 1 < len(rows_data):
                                dr = rows_data[j + 1]
                                if dr[year_col] == report_year and dr[val_col] is not None:
                                    try:
                                        val = int(dr[val_col])
                                        if val > 0:
                                            ev_records.append((
                                                report_date, "Osobní automobily", brand_name,
                                                pt, val
                                            ))
                                            metrics_found += 1
                                    except (ValueError, TypeError):
                                        pass
                            if metrics_found >= 2:  # BEV + PHEV only
                                break
                    break  # Only first monthly section for this brand

    print(f"  {filename}: {len(ev_records):,} total records")

print(f"\nTotal EV records: {len(ev_records):,}")

In [0]:
# Write EV production table (per brand only, no Celkem)
ev_schema = StructType([
    StructField("report_date", DateType(), False),
    StructField("vehicle_category", StringType(), False),
    StructField("brand", StringType(), False),
    StructField("powertrain", StringType(), False),
    StructField("value", IntegerType(), False),
])

df_ev = spark.createDataFrame(ev_records, schema=ev_schema)
df_ev = df_ev.dropDuplicates()

df_ev.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE_EV)

spark.sql(f"""COMMENT ON TABLE {TARGET_TABLE_EV} IS
    'Monthly electric vehicle production in Czech Republic PER BRAND by powertrain type (BEV, PHEV). No aggregated totals - only individual manufacturer data (ŠKODA AUTO, Hyundai). Source: SAP (Sdružení automobilového průmyslu).'""")
spark.sql(f"ALTER TABLE {TARGET_TABLE_EV} ALTER COLUMN report_date COMMENT 'First day of the reporting month'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_EV} ALTER COLUMN vehicle_category COMMENT 'Vehicle category'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_EV} ALTER COLUMN brand COMMENT 'Manufacturer (ŠKODA AUTO, Hyundai)'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_EV} ALTER COLUMN powertrain COMMENT 'Powertrain type: BEV (battery electric), PHEV (plug-in hybrid)'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_EV} ALTER COLUMN value COMMENT 'Number of vehicles produced'")

print(f"\u2705 {TARGET_TABLE_EV}: {spark.table(TARGET_TABLE_EV).count():,} rows")
print(f"\nBrand x Powertrain breakdown:")
display(spark.table(TARGET_TABLE_EV).groupBy("brand", "powertrain").count().orderBy("brand", "powertrain"))
print(f"\nSample data:")
display(spark.table(TARGET_TABLE_EV).orderBy(F.desc("report_date"), "brand", "powertrain").limit(10))

In [0]:
# Summary
print("=" * 60)
print("INGESTION COMPLETE")
print("=" * 60)

for table_name in [TARGET_TABLE_YEARLY, TARGET_TABLE_MONTHLY, TARGET_TABLE_EV]:
    count = spark.table(table_name).count()
    print(f"  {table_name}: {count:,} rows")

print("\n=== Monthly production sample (latest) ===")
display(
    spark.table(TARGET_TABLE_MONTHLY)
    .orderBy(F.desc("report_date"), "brand", "metric")
    .limit(15)
)

print("\n=== EV production sample (latest) ===")
display(
    spark.table(TARGET_TABLE_EV)
    .orderBy(F.desc("report_date"), "brand", "powertrain")
    .limit(10)
)